# Ask-SLM CLI Runner (No UI)

This notebook runs Ask-SLM CLI capabilities using a selected workspace only.

Usage flow:
1. Set `WORKSPACE` in Cell 2.
2. Run one or more analysis cells below.
3. Outputs are written into `Workspaces/<WORKSPACE>/results/` by the CLI defaults.

In [10]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Configure this once per run.
WORKSPACE = "example_workspace"#"ABX"
WORKSPACE_ROOT = Path("./Workspaces")
# Resolve repo root from the notebook's parent directory
REPO_ROOT = Path("..").resolve()


def run_cli(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    """Run askslm.py with pretty command logging, from the repo root."""
    cmd = [sys.executable, "askslm.py"] + args
    print("$", " ".join(shlex.quote(part) for part in cmd))
    result = subprocess.run(cmd, text=True, cwd=str(REPO_ROOT))
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


workspace_dir = WORKSPACE_ROOT / WORKSPACE
workspace_inputs = workspace_dir / "inputs"
workspace_configs = workspace_dir / "mapping_configs"
workspace_results = workspace_dir / "results"

print(f"Workspace: {workspace_dir}")
print(f"Inputs:    {workspace_inputs}")
print(f"Configs:   {workspace_configs}")
print(f"Results:   {workspace_results}")

run_cli([
    "init",
    "--workspace", WORKSPACE,
    "--location", str(WORKSPACE_ROOT),
])


Workspace: Workspaces/example_workspace
Inputs:    Workspaces/example_workspace/inputs
Configs:   Workspaces/example_workspace/mapping_configs
Results:   Workspaces/example_workspace/results
$ /Users/yarintamam/Code/Ask-SLM/.venv/bin/python askslm.py init --workspace example_workspace --location Workspaces
Workspace initialization complete
  Workspace: Workspaces/example_workspace
  Created:
    - Workspaces/example_workspace
    - Workspaces/example_workspace/inputs
    - Workspaces/example_workspace/inputs/metadata_parts
    - Workspaces/example_workspace/inputs/taxonomy_parts
    - Workspaces/example_workspace/inputs/appendix


CompletedProcess(args=['/Users/yarintamam/Code/Ask-SLM/.venv/bin/python', 'askslm.py', 'init', '--workspace', 'example_workspace', '--location', 'Workspaces'], returncode=0)

In [ ]:
# Metadata overoverview
run_cli([
    "metadata-overview",
    "--workspace", WORKSPACE,
])

In [ ]:
# Abundance matrix
run_cli([
    "abundance-matrix",
    "--workspace", WORKSPACE,
])

In [ ]:
# Build taxonomy lineage DB (required by alpha-diversity)
run_cli([
    "build-taxonomy-lineage",
    "--workspace", WORKSPACE,
])

In [ ]:
# Alpha diversity
ALPHA_RANK = "genus"
ALPHA_GROUP = None  # Example: "Group"
LINEAGE_DB_PATH = workspace_results / "_cache" / "taxonomy_lineage_db.json"

cmd = [
    "alpha-diversity",
    "--workspace", WORKSPACE,
    "--rank", ALPHA_RANK,
    "--lineage-db-path", str(LINEAGE_DB_PATH),
]
if ALPHA_GROUP:
    cmd += ["--group", ALPHA_GROUP]

run_cli(cmd)

In [ ]:
# Taxonomic abundance
TAXONOMIC_RANK = "genus"
TAXONOMIC_TAXON = "Prevotella"
TAXONOMIC_GROUP = None  # Example: "Group"

cmd = [
    "taxonomic-abundance",
    "--workspace", WORKSPACE,
    "--rank", TAXONOMIC_RANK,
    "--taxon", TAXONOMIC_TAXON,
]
if TAXONOMIC_GROUP:
    cmd += ["--group", TAXONOMIC_GROUP]

run_cli(cmd)

In [ ]:
# Single taxon abundance
SINGLE_TAXON_RANK = "superkingdom"
SINGLE_TAXON_NAME = "Viruses"
SINGLE_SOURCE_KEYS = ["sample_name", "host_subject_id_sam"]

run_cli([
    "single-taxon-abundance",
    "--workspace", WORKSPACE,
    "--rank", SINGLE_TAXON_RANK,
    "--taxon", SINGLE_TAXON_NAME,
    "--source-keys", *SINGLE_SOURCE_KEYS,
])